# MIT–BIT charging controller benchmark

Compare fixed 1C, fixed 2C, fixed 3C, a past-observation rule and the Notebook 15 energy model. Each policy chooses the rate for the next five cycles using the history through the last completed cycle. The target is 90% of the previous block's delivered energy.

The dataset randomized the logged rate across 1C, 2C and 3C. A policy gets a measured outcome when its action matches the logged action. This is a one-block benchmark on recorded histories, with held-out batteries and battery-level uncertainty.

Run all cells from the LifeGuard-RL project using the same environment as Notebook 15. Inputs are Notebook 14 and 15 report folders or ZIPs and the MIT–BIT v3 aging workbooks. One CPU thread is sufficient. The timing audit reads the workbooks sequentially.

In [1]:
import os
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[name] = "1"

import json
import re
from collections import defaultdict
from contextlib import contextmanager
from pathlib import Path, PurePosixPath
from tempfile import SpooledTemporaryFile
from xml.etree import ElementTree
from zipfile import ZIP_DEFLATED, ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from threadpoolctl import threadpool_limits

DATASET_PATH = None
NOTEBOOK_14_EXPORT = None
NOTEBOOK_15_EXPORT = None
OUTPUT_ROOT = None

PREVIOUS_ENERGY_FRACTION = 0.90
OVERPREDICTION_QUANTILE = 0.90
ALLOWED_HIT_DROP = 0.05
BOOTSTRAP_DRAWS = 2000
SEED = 42
FIRST_START, LAST_START, BLOCK = 11, 91, 5
LAST_AGING_CYCLE = LAST_START + BLOCK - 1
NOMINAL_AH = 2.4
RATES = (1, 2, 3)
FOLDS = (1, 2, 3, 4, 5)
LOG_PROBABILITY = 1 / len(RATES)
KEYS = ["cell_id", "start_cycle"]
POLICIES = {
    "fixed_1c": "Fixed 1C",
    "fixed_2c": "Fixed 2C",
    "fixed_3c": "Fixed 3C",
    "rule": "Past-observation rule",
    "model": "Model-guided",
}
OUTPUT_NAME = "lifeguard_mit_bit_controller_benchmark_results"

cwd = Path.cwd()
project = next((p for p in (cwd, *cwd.parents)
                if (p / "README_LifeGuard_RL.md").is_file() and (p / "reports").is_dir()), None)
if project is None:
    raise FileNotFoundError("Open this notebook from within the LifeGuard-RL project.")


def input_path(override, default):
    return Path(override).expanduser() if override is not None else (
        default if default.is_dir() else default.with_suffix(".zip"))


raw_source = input_path(DATASET_PATH, project / "Dataset" / "MIT-BIT")
source14 = input_path(NOTEBOOK_14_EXPORT, project / "reports" / "lifeguard_mit_bit_switch_replay_results")
source15 = input_path(NOTEBOOK_15_EXPORT, project / "reports" / "lifeguard_mit_bit_capacity_energy_results")
output = (Path(OUTPUT_ROOT).expanduser() if OUTPUT_ROOT is not None else project / "reports") / OUTPUT_NAME


def read_tables(path, names):
    if path.is_dir():
        return {name: pd.read_csv(path / f"{name}.csv") for name in names}
    if not path.is_file():
        raise FileNotFoundError(f"Missing prerequisite report: {path}")
    with ZipFile(path) as bundle:
        tables = {}
        for name in names:
            files = [file for file in bundle.namelist() if PurePosixPath(file).name == f"{name}.csv"]
            if len(files) != 1:
                raise ValueError(f"Expected one {name}.csv in {path}.")
            with bundle.open(files[0]) as stream:
                tables[name] = pd.read_csv(stream)
        return tables


def require_unique(table, keys, label):
    if table[keys].isna().any().any() or table.duplicated(keys).any():
        raise ValueError(f"Missing or duplicate {keys} in {label}.")

## Cohort and decision history

Eligible starts are cycles 11 through 91, with a previous block whose Wh is measured and a verified logged rate. The final scheduled block, cycles 96–100, is outside the measurement window used here. Both the start and the end of the eligible window are fixed before policy scoring. Missing next-block outcomes stay in the replay denominators and coverage audit.

The feature construction and energy model use Notebook 15's two Ridge components. Each candidate rate changes the action flags and the associated same-rate history. Production features consume completed blocks only.

In [2]:
def audited_blocks(blocks, cycles):
    require_unique(blocks, KEYS, "Notebook 14 block audit")
    require_unique(cycles, ["cell_id", "aging_cycle"], "Notebook 14 cycle audit")
    if not blocks.end_cycle.eq(blocks.start_cycle + BLOCK - 1).all():
        raise ValueError("Expected five-cycle blocks in Notebook 14.")

    raw = cycles.set_index(["cell_id", "aging_cycle"]).sort_index()
    observations = []
    for row in blocks.itertuples(index=False):
        window = raw.loc[row.cell_id].reindex(range(row.start_cycle, row.end_cycle + 1))
        capacity = window.capacity_ah
        delivered = window.delivered_wh
        equivalent_ah = window.integrated_ah
        # Score the same measured blocks as Notebook 14. The audit supplies their physical units.
        observed_ah = capacity.median() if capacity.notna().all() and pd.notna(row.observed_soh_pct) else np.nan
        observed_wh = delivered.sum() if delivered.notna().all() else np.nan
        total_ah = equivalent_ah.sum() if equivalent_ah.notna().all() and pd.notna(observed_wh) else np.nan
        observations.append((observed_ah, observed_wh, total_ah))
    table = blocks.copy()
    table[["observed_ah", "measured_wh", "total_integrated_ah"]] = observations
    np.testing.assert_allclose(table.observed_ah, table.observed_soh_pct * table.reference_ah / 100,
                               rtol=0, atol=1e-9)
    np.testing.assert_allclose(table.measured_wh, table.observed_wh, rtol=0, atol=1e-9)
    return table, raw.capacity_ah


def validate_folds(events, splits):
    require_unique(events, KEYS, "Notebook 14 predictions")
    require_unique(splits, ["fold", "cell_id"], "Notebook 14 cell splits")
    if set(splits.fold) != set(range(1, 6)) or events.cell_id.nunique() != 55:
        raise ValueError("Notebook 15 expects the full 55-cell, five-fold Notebook 14 benchmark.")
    test_fold = splits.loc[splits.role.eq("test")].set_index("cell_id", verify_integrity=True).fold
    cells = set(events.cell_id)
    if set(test_fold.index) != cells or not events.cell_id.map(test_fold).eq(events.fold).all():
        raise ValueError("Notebook 14 prediction rows and held-out folds disagree.")
    expected = {(fold, cell, "test" if test_fold[cell] == fold else "train")
                for fold in range(1, 6) for cell in cells}
    if set(splits[["fold", "cell_id", "role"]].itertuples(index=False, name=None)) != expected:
        raise ValueError("The saved train/test split is incomplete.")


def causal_features(events, blocks, previous_capacity):
    records = []
    last_energy = []
    for cell, selected in events.groupby("cell_id", sort=False):
        trace = blocks.loc[blocks.cell_id.eq(cell)].set_index("start_cycle").sort_index()
        valid = trace.loc[trace.status.eq("ok")]
        first_rate = int(trace.charge_rate_c.iloc[0])
        for row in selected.itertuples(index=True):
            prior = valid.loc[valid.index < row.start_cycle]
            current, older = [prior.loc[row.start_cycle - offset]
                              for offset in (BLOCK, 2 * BLOCK)]
            if int(current.charge_rate_c) != int(row.previous_rate_c):
                raise ValueError(f"Incorrect source rate for cell {cell}, cycle {row.start_cycle}.")
            latest = previous_capacity.get((cell, row.start_cycle - 1), np.nan)
            block_start = previous_capacity.get((cell, row.start_cycle - BLOCK), np.nan)
            detail = {
                "start_ah": current.observed_ah, "latest_ah": latest,
                "recent_block_delta_ah": current.observed_ah - older.observed_ah,
                "within_block_delta_ah": latest - block_start if np.isfinite(latest) and np.isfinite(block_start) else 0.0,
                "age_cycles": row.start_cycle - 1, "first_rate_c": first_rate,
                "prior_2c_blocks": int(prior.charge_rate_c.eq(2).sum()),
                "prior_3c_blocks": int(prior.charge_rate_c.eq(3).sum()),
            }
            for rate in RATES:
                matched = prior.loc[prior.charge_rate_c.eq(rate) & prior.observed_ah.notna()].tail(3)
                last = float(matched.observed_ah.iloc[-1]) if len(matched) else np.nan
                gap = row.start_cycle - matched.index[-1] if len(matched) else np.nan
                slope = ((last - matched.observed_ah.iloc[0]) / (matched.index[-1] - matched.index[0])
                         if len(matched) > 1 else 0.0)
                detail[f"last_{rate}c_ah"] = last
                detail[f"lag_{rate}c_cycles"] = gap
                detail[f"projected_{rate}c_ah"] = min(last, last + gap * slope) if len(matched) else np.nan
            energy = {f"last_{rate}c_wh": prior.loc[prior.charge_rate_c.eq(rate), "observed_wh"]
                      .dropna().iloc[-1:].sum(min_count=1) for rate in RATES}
            records.append(pd.Series(detail, name=row.Index))
            last_energy.append(pd.Series(energy, name=row.Index))
    return (pd.DataFrame(records).reindex(events.index),
            pd.DataFrame(last_energy).reindex(events.index))



audit = read_tables(source14, ("block_audit", "cycle_audit", "predictions", "cell_splits", "source_audit"))
reference = read_tables(source15, ("predictions",))["predictions"]
events = audit["predictions"].sort_values(KEYS).reset_index(drop=True)
validate_folds(events, audit["cell_splits"])
require_unique(reference, KEYS, "Notebook 15 predictions")
reference = reference.set_index(KEYS).reindex(pd.MultiIndex.from_frame(events[KEYS]))
if (not reference.energy_model.eq("blend_75_25").all()
        or not np.array_equal(reference.charge_rate_c, events.charge_rate_c)
        or not np.array_equal(reference.fold, events.fold)):
    raise ValueError("Expected Notebook 15's 75:25 energy blend on the original five held-out folds.")

blocks, raw_capacity = audited_blocks(audit["block_audit"], audit["cycle_audit"])
aligned = blocks.set_index(KEYS).reindex(pd.MultiIndex.from_frame(events[KEYS]))
history, last_energy = causal_features(events, blocks, raw_capacity)
np.testing.assert_allclose(history.start_ah, reference.start_ah, rtol=0, atol=1e-9)
np.testing.assert_allclose(events.observed_wh, reference.observed_wh, rtol=0, atol=1e-9)
np.testing.assert_allclose(events.start_wh, reference.start_wh, rtol=0, atol=1e-9)
previous_keys = pd.MultiIndex.from_frame(events[KEYS].assign(start_cycle=events.start_cycle - BLOCK))
previous_wh = blocks.set_index(KEYS).observed_wh.reindex(previous_keys)
np.testing.assert_allclose(events.start_wh, previous_wh, rtol=0, atol=1e-9)
np.testing.assert_allclose(aligned.observed_ah, reference.observed_ah, rtol=0, atol=1e-9)
if not events.charge_rate_c.isin(RATES).all() or not events.end_cycle.eq(events.start_cycle + BLOCK - 1).all():
    raise ValueError("Expected verified 1C/2C/3C five-cycle actions.")

window = events.start_cycle.between(FIRST_START, LAST_START)
eligible = window & events.start_wh.gt(0) & np.isfinite(events.start_wh)
cohort = events.loc[eligible, [*KEYS, "end_cycle", "fold", "previous_rate_c", "charge_rate_c",
                               "start_wh", "observed_wh"]].copy()
cohort["observed_ah"] = aligned.observed_ah.to_numpy()[eligible]
cohort["target_wh"] = PREVIOUS_ENERGY_FRACTION * cohort.start_wh
print(f"{len(cohort)} eligible origins on {cohort.cell_id.nunique()} batteries; "
      f"{int((window & ~eligible).sum())} excluded for missing previous Wh; "
      f"{cohort.observed_wh.isna().sum()} eligible origins with missing next Wh.")

929 eligible origins on 55 batteries; 6 excluded for missing previous Wh; 6 eligible origins with missing next Wh.


## Measured charging duration

For each aging cycle the duration is the elapsed time between the first and last positive-current sample above 0.1 A in the charging segment. The check requires a verified rate, an observed end to the charging segment and consistent timestamps. For discontinuous wall-clock timestamps, use the same trace's elapsed Test_Time after validating every charge sample. This separates active charging time from logger pauses. A block has a measured duration when all five cycles pass. Rest and discharge time are outside the metric.

In [3]:
SHEET = "xl/worksheets/sheet1.xml"
HEADERS = ("Data_Point", "Test_Time(s)", "Current(A)", "Capacity(Ah)", "Voltage(V)",
           "Energy(Wh)", "Temperature(℃)", "Date_Time", "Cycle_Index")
FIELDS = b"CDEFHI"
CELL_VALUE = re.compile(rb'<c r="([A-I])(\d+)"[^>]*><v>([^<]*)</v></c>')
NUMERIC = re.compile(rb"[+\-0-9.Ee]+")
STRING = re.compile(rb"<si(?:\s[^>]*)?>.*?</si>", re.DOTALL)



def validate_header(workbook, row):
    entries = CELL_VALUE.findall(row)
    if len(entries) != len(HEADERS):
        raise ValueError("Unexpected MIT–BIT worksheet header layout.")
    table = workbook.read("xl/sharedStrings.xml")
    wanted = {int(value) for _, _, value in entries}
    labels = {}
    for index, match in enumerate(STRING.finditer(table)):
        if index in wanted:
            labels[index] = "".join(ElementTree.fromstring(match[0]).itertext())
            if len(labels) == len(wanted):
                break
    if tuple(labels.get(int(value)) for _, _, value in entries) != HEADERS:
        raise ValueError("Unexpected MIT–BIT worksheet columns or units.")


def trace_chunks(workbook):
    # Stream the verified worksheet instead of building a workbook object with millions of cells.
    with workbook.open(SHEET) as xml:
        pending = xml.read(2**20)
        header_end = pending.find(b"</row>")
        if header_end < 0:
            raise ValueError("Missing worksheet header.")
        validate_header(workbook, pending[:header_end])
        pending = pending[header_end + len(b"</row>"):]
        while pending:
            cutoff = pending.rfind(b"</row>")
            if cutoff < 0:
                if b"</sheetData>" in pending:
                    break
                chunk = xml.read(2**20)
                if not chunk:
                    raise ValueError("Incomplete worksheet XML.")
                pending += chunk
                continue
            cutoff += len(b"</row>")
            complete, pending = pending[:cutoff], pending[cutoff:]
            values = [(col, value) for col, _, value in CELL_VALUE.findall(complete)
                      if col[0] in FIELDS]
            rows = complete.count(b"</row>")
            if (len(values) != rows * len(FIELDS)
                    or any(col[0] != FIELDS[i % len(FIELDS)] or not NUMERIC.fullmatch(value)
                           for i, (col, value) in enumerate(values))):
                raise ValueError("Incomplete numeric fields or reordered columns in the aging worksheet.")
            sample = np.fromstring(b" ".join(value for _, value in values), sep=" ").reshape(-1, len(FIELDS))
            yield sample
            pending += xml.read(2**20)


def cycle_traces(workbook):
    carry = np.empty((0, len(FIELDS)))
    previous = 0
    for chunk in trace_chunks(workbook):
        data = np.concatenate((carry, chunk))
        split = np.flatnonzero(data[:-1, -1] != data[1:, -1]) + 1
        groups = np.split(data, split)
        for trace in groups[:-1]:
            raw = trace[0, -1]
            if raw <= previous or raw != int(raw):
                raise ValueError("Repeated or unordered raw cycle index.")
            previous = int(raw)
            yield previous, trace
        carry = groups[-1]
    if carry.size:
        raw = carry[0, -1]
        if raw <= previous or raw != int(raw):
            raise ValueError("Repeated or unordered final cycle index.")
        yield int(raw), carry



@contextmanager
def open_aging_workbook(path, archive, name, zip_paths):
    if archive is None:
        with ZipFile(path / name) as workbook:
            yield workbook
        return
    with archive.open(zip_paths[name]) as compressed, SpooledTemporaryFile(max_size=4 * 2**20) as stream:
        while chunk := compressed.read(2**20):
            stream.write(chunk)
        stream.seek(0)
        with ZipFile(stream) as workbook:
            yield workbook


TEST_TIME_CELL = re.compile(rb'<c r="B(\d+)"([^>]*)><v>([^<]+)</v></c>')
ELAPSED_TIME = re.compile(r"(?:(\d+)-)?(\d{1,2}):(\d{2}):(\d{2})")


def test_time_seconds(workbook, first_row, last_row):
    entries = {}
    with workbook.open(SHEET) as xml:
        pending = b""
        while chunk := xml.read(2**20):
            pending += chunk
            cutoff = pending.rfind(b"</row>")
            if cutoff < 0:
                continue
            cutoff += len(b"</row>")
            for match in TEST_TIME_CELL.finditer(pending[:cutoff]):
                row = int(match[1])
                if first_row <= row <= last_row:
                    entries[row] = (b't="s"' in match[2], match[3])
                if row >= last_row:
                    break
            if last_row in entries:
                break
            pending = pending[cutoff:]
    if sorted(entries) != list(range(first_row, last_row + 1)):
        return np.full(last_row - first_row + 1, np.nan)

    wanted = {int(value) for is_string, value in entries.values() if is_string}
    labels = {}
    if wanted:
        for index, match in enumerate(STRING.finditer(workbook.read("xl/sharedStrings.xml"))):
            if index in wanted:
                labels[index] = "".join(ElementTree.fromstring(match[0]).itertext())
                if len(labels) == len(wanted):
                    break

    seconds = []
    for row in range(first_row, last_row + 1):
        is_string, value = entries[row]
        if not is_string:
            # The workbook formats numeric elapsed times as fractions of an Excel day.
            seconds.append(float(value) * 86400)
            continue
        clock = ELAPSED_TIME.fullmatch(labels.get(int(value), ""))
        if clock is None:
            return np.full(last_row - first_row + 1, np.nan)
        days, hour, minute, second = (int(part or 0) for part in clock.groups())
        seconds.append(days * 86400 + hour * 3600 + minute * 60 + second)
    return np.asarray(seconds)


def valid_clock(seconds):
    return (np.isfinite(seconds).all() and np.all(np.diff(seconds) >= -1.1)
            and np.all(np.diff(np.maximum.accumulate(seconds)) <= 30))


def charge_duration(trace, expected_rate, workbook, first_sheet_row):
    current, _, _, _, date_days, _ = trace.T
    if not np.isfinite(current).all() or not np.isfinite(date_days).all():
        return np.nan, np.nan, "invalid_trace"
    negative = np.flatnonzero(current < -0.1)
    stop = negative[0] if negative.size else len(current)
    positive = np.flatnonzero(current[:stop] > 0.1)
    if positive.size < 5 or positive[-1] == len(current) - 1 or np.any(np.diff(positive) != 1):
        return np.nan, np.nan, "incomplete_or_interrupted_charge"
    samples = current[positive]
    amps = NOMINAL_AH * expected_rate
    if (abs(np.quantile(samples, 0.9) - amps) > 0.05 * amps
            or np.sum(abs(samples - amps) <= 0.05 * amps) < 5):
        raise ValueError(f"Raw workbook does not verify the audited {expected_rate}C charge.")
    wall_clock = (date_days[positive] - date_days[positive[0]]) * 86400
    wall_minutes = float(wall_clock[-1] / 60)
    seconds, status = wall_clock, "ok_wall_clock"
    if not valid_clock(wall_clock):
        start, end = first_sheet_row + positive[[0, -1]]
        elapsed = test_time_seconds(workbook, start, end)
        if not valid_clock(elapsed):
            return np.nan, wall_minutes, "invalid_timestamps"
        seconds, status = elapsed - elapsed[0], "ok_test_time"
    duration = float(np.maximum.accumulate(seconds)[-1] / 60)
    return (duration, wall_minutes, status) if duration > 0 else (np.nan, wall_minutes, "invalid_timestamps")


def measure_charges(path, workbook_audit, decisions):
    if not path.exists():
        raise FileNotFoundError(f"Missing MIT–BIT workbooks: {path}. Set DATASET_PATH if they moved.")
    workbook_audit = workbook_audit.loc[workbook_audit.status.eq("read")]
    require_unique(workbook_audit, ["cell_id"], "Notebook 14 workbook audit")
    workbook_paths = workbook_audit.set_index("cell_id").workbook
    if not set(decisions.cell_id).issubset(workbook_paths.index):
        raise ValueError("A benchmark battery is missing from Notebook 14's workbook audit.")

    archive = None if path.is_dir() else ZipFile(path)
    records = []
    try:
        zip_paths = {}
        if archive is not None:
            by_suffix = defaultdict(list)
            for file in archive.namelist():
                by_suffix["/".join(PurePosixPath(file).parts[-2:])].append(file)
            for name in workbook_paths:
                matches = by_suffix.get(name, [])
                if len(matches) != 1:
                    raise ValueError(f"Expected one aging workbook ending in {name} in {path}.")
                zip_paths[name] = matches[0]
        grouped = decisions.groupby("cell_id", sort=True)
        for position, (cell, items) in enumerate(grouped, start=1):
            pending = {aging + 1: (start, rate)
                       for start, rate in items[["start_cycle", "charge_rate_c"]].itertuples(index=False, name=None)
                       for aging in range(start, start + BLOCK)}
            with open_aging_workbook(path, archive, workbook_paths[cell], zip_paths) as workbook:
                next_sheet_row = 2
                for raw, trace in cycle_traces(workbook):
                    first_sheet_row = next_sheet_row
                    next_sheet_row += len(trace)
                    if raw > LAST_AGING_CYCLE + 1:
                        break
                    if raw in pending:
                        start, rate = pending.pop(raw)
                        try:
                            minutes, wall_minutes, status = charge_duration(trace, rate, workbook, first_sheet_row)
                        except ValueError as error:
                            raise ValueError(f"Cell {cell}, raw cycle {raw}. {error}") from error
                        records.append((cell, start, raw - 1, rate, minutes, wall_minutes, status))
            for raw, (start, rate) in sorted(pending.items()):
                records.append((cell, start, raw - 1, rate, np.nan, np.nan, "missing_cycle"))
            if position % 10 == 0 or position == grouped.ngroups:
                print(f"Read charge timing for {position}/{grouped.ngroups} batteries", flush=True)
    finally:
        if archive is not None:
            archive.close()
    return pd.DataFrame(records, columns=[*KEYS, "aging_cycle", "charge_rate_c", "charge_minutes", "wall_clock_minutes", "status"])

In [4]:
with threadpool_limits(limits=1):
    charge_audit = measure_charges(raw_source, audit["source_audit"], cohort)
require_unique(charge_audit, ["cell_id", "aging_cycle"], "charge timing audit")
time_by_block = charge_audit.groupby(KEYS).charge_minutes.agg(
    measured_cycles="count", charge_minutes=lambda values: values.sum(min_count=BLOCK))
cohort = cohort.join(time_by_block, on=KEYS, validate="one_to_one")
if len(charge_audit) != BLOCK * len(cohort) or cohort.measured_cycles.isna().any():
    raise ValueError("Charging durations are not aligned to the decision cohort.")
print(f"Complete charging times for {cohort.charge_minutes.notna().sum()}/{len(cohort)} blocks. "
      f"Used Test_Time for {charge_audit.status.eq('ok_test_time').sum()} cycles with wall-clock discontinuities.")

Read charge timing for 10/55 batteries
Read charge timing for 20/55 batteries
Read charge timing for 30/55 batteries
Read charge timing for 40/55 batteries
Read charge timing for 50/55 batteries
Read charge timing for 55/55 batteries
Complete charging times for 929/929 blocks. Used Test_Time for 10 cycles with wall-clock discontinuities.


## Five policies and fold-local calibration

Every rate is evaluated under the same decision history. The model uses a 75:25 blend of the Notebook 15 standard and expanded Ridge models with alpha 1 in both components. The risk margin is the nonnegative 90th percentile of overprediction among inner out-of-fold training forecasts for that rate. The code verifies that the model reproduces the Notebook 15 forecasts for the logged rates.

The rule compares the latest measured Wh at each previously observed rate with the target. The model compares its estimated Wh minus the margin. Both use median measured charging duration by rate on outer training batteries to order the feasible rates. If no rate clears the target, the rule uses the highest available historical Wh and the model uses the highest conservative forecast. With no candidate history, the rule chooses 1C.

In [5]:
def design_matrices(rows):
    next_rate = rows.charge_rate_c.to_numpy()
    previous = rows.previous_rate_c.to_numpy()
    rate_flags = np.column_stack([(next_rate == rate) for rate in (2, 3)]
                                 + [(previous == rate) for rate in (2, 3)])
    transition_flags = np.column_stack([(previous == a) & (next_rate == b)
                                        for a in RATES for b in RATES if (a, b) != (1, 1)])
    start = rows.start_ah.to_numpy()
    history = rows[["recent_block_delta_ah", "within_block_delta_ah", "age_cycles",
                    "prior_3c_blocks", "prior_2c_blocks"]].to_numpy()
    common = np.column_stack((start, rows.latest_ah, rate_flags, transition_flags, history))
    interactions = start[:, None] * rate_flags
    same = rows[["last_same_rate_ah", "projected_same_rate_ah", "lag_same_rate_cycles"]].to_numpy()
    per_rate = rows[[*(f"last_{rate}c_ah" for rate in RATES),
                     *(f"projected_{rate}c_ah" for rate in RATES),
                     *(f"lag_{rate}c_cycles" for rate in RATES)]].to_numpy()
    return {
        "standard": np.column_stack((common, interactions, history[:, :2] * rate_flags[:, :2])),
        "expanded": np.column_stack((common, same, per_rate, interactions)),
    }


def ridge_predict(a, b, labels, sample):
    weights = 1 / sample.groupby(sample).transform("size").to_numpy(dtype=float)
    weights /= weights.mean()
    usable = np.isfinite(a).any(axis=0)
    a, b = a[:, usable], b[:, usable]
    missing_a, missing_b = ~np.isfinite(a), ~np.isfinite(b)
    indicators = missing_a.any(axis=0)
    medians = np.nanmedian(np.where(missing_a, np.nan, a), axis=0)
    a = np.column_stack((np.where(missing_a, medians, a), missing_a[:, indicators]))
    b = np.column_stack((np.where(missing_b, medians, b), missing_b[:, indicators]))
    center = np.average(a, axis=0, weights=weights)
    scale = np.sqrt(np.average((a - center)**2, axis=0, weights=weights))
    scale = np.maximum(scale, 1e-8)
    fit = Ridge(alpha=1)
    fit.fit((a - center) / scale, labels, sample_weight=weights)
    return fit.predict((b - center) / scale)


def action_features(history, previous_rate, candidate):
    table = history.copy()
    table["previous_rate_c"] = previous_rate
    table["charge_rate_c"] = candidate
    for feature, suffix in (("last", "ah"), ("projected", "ah"), ("lag", "cycles")):
        history_by_rate = table[[f"{feature}_{rate}c_{suffix}" for rate in RATES]].to_numpy()
        rate_index = np.broadcast_to(np.asarray(candidate, dtype=int), len(table)) - 1
        table[f"{feature}_same_rate_{suffix}"] = history_by_rate[np.arange(len(table)), rate_index]
    return design_matrices(table)


def energy_forecast(training, test, labels, batteries):
    standard = ridge_predict(training["standard"], test["standard"], labels, batteries)
    expanded = ridge_predict(training["expanded"], test["expanded"], labels, batteries)
    return np.maximum(0, 0.75 * standard + 0.25 * expanded)


def subset(design, mask):
    return {name: matrix[mask] for name, matrix in design.items()}


def choose_rate(scores, typical_minutes, target_wh):
    speeds = np.broadcast_to(typical_minutes, scores.shape)
    known = np.isfinite(scores)
    feasible = known & (scores >= target_wh[:, None])
    # Rates are ordered 1C, 2C, 3C, so a tie favors the lower rate.
    quickest = np.where(feasible, speeds, np.inf).argmin(axis=1)
    best_available = np.where(known, scores, -np.inf).argmax(axis=1)
    fallback = ~feasible.any(axis=1)
    return np.asarray(RATES)[np.where(fallback, best_available, quickest)], fallback


def forecast_actions(events, history, cohort, reference):
    logged = action_features(history, events.previous_rate_c, events.charge_rate_c)
    candidates = {rate: action_features(history, events.previous_rate_c, rate) for rate in RATES}
    labels = events.observed_wh.to_numpy()
    usable = np.isfinite(labels) & events.start_wh.notna().to_numpy()
    folds = events.fold.to_numpy()
    logged_rates = events.charge_rate_c.to_numpy()
    results = []
    calibration = []
    baseline = np.full(len(events), np.nan)
    for outer in FOLDS:
        test = folds == outer
        train = ~test & usable
        residuals = np.full(len(events), np.nan)
        for inner in FOLDS:
            if inner == outer:
                continue
            fit = train & (folds != inner)
            validate = train & (folds == inner)
            residuals[validate] = (energy_forecast(subset(logged, fit), subset(logged, validate),
                                                  labels[fit], events.cell_id[fit]) - labels[validate])
        if not np.isfinite(residuals[train]).all():
            raise ValueError(f"Incomplete inner energy predictions for fold {outer}.")
        baseline[test] = energy_forecast(subset(logged, train), subset(logged, test),
                                         labels[train], events.cell_id[train])
        train_times = (cohort.loc[cohort.fold.ne(outer)].groupby("charge_rate_c")
                       .charge_minutes.median().reindex(RATES))
        if not np.isfinite(train_times).all():
            raise ValueError(f"Missing training charging durations for fold {outer}.")
        outer_cohort = cohort.index[cohort.fold.eq(outer)]
        for rate in RATES:
            samples = residuals[train & (logged_rates == rate)]
            if not len(samples):
                raise ValueError(f"No inner residuals for fold {outer}, rate {rate}C.")
            margin = max(0.0, float(np.quantile(samples, OVERPREDICTION_QUANTILE)))
            typical = float(train_times.loc[rate])
            estimate = energy_forecast(subset(logged, train), subset(candidates[rate], outer_cohort),
                                       labels[train], events.cell_id[train])
            frame = cohort.loc[outer_cohort, [*KEYS, "fold", "target_wh"]].copy()
            frame["candidate_rate_c"] = rate
            frame["forecast_wh"] = estimate
            frame["overprediction_margin_wh"] = margin
            frame["conservative_wh"] = np.maximum(0, estimate - margin)
            frame["training_typical_charge_minutes"] = typical
            results.append(frame)
            calibration.append((outer, rate, len(samples), margin, typical))
    np.testing.assert_allclose(baseline, reference.predicted_wh, rtol=0, atol=1e-9)
    return (pd.concat(results).sort_values([*KEYS, "candidate_rate_c"], ignore_index=True),
            pd.DataFrame(calibration, columns=["fold", "candidate_rate_c", "inner_residuals",
                                              "overprediction_margin_wh", "training_typical_charge_minutes"]))


def replay_decisions(cohort, forecasts, last_energy):
    origin_index = pd.MultiIndex.from_frame(cohort[KEYS])
    target = cohort.target_wh.to_numpy()
    matrices = {column: forecasts.pivot(index=KEYS, columns="candidate_rate_c", values=column)
                .reindex(index=origin_index, columns=RATES).to_numpy()
                for column in ("conservative_wh", "training_typical_charge_minutes")}
    prior_wh = last_energy.loc[cohort.index].to_numpy()
    predictions = matrices["conservative_wh"]
    speeds = matrices["training_typical_charge_minutes"]
    forecasts["rule_prior_wh"] = forecasts.join(
        last_energy.join(cohort[KEYS]).dropna(subset=KEYS).set_index(KEYS), on=KEYS,
        validate="many_to_one")[[f"last_{rate}c_wh" for rate in RATES]].to_numpy()[
            np.arange(len(forecasts)), forecasts.candidate_rate_c.to_numpy() - 1]
    chosen = {f"fixed_{rate}c": (np.full(len(cohort), rate), np.zeros(len(cohort), dtype=bool)) for rate in RATES}
    chosen["rule"] = choose_rate(prior_wh, speeds, target)
    chosen["model"] = choose_rate(predictions, speeds, target)
    baseline = cohort.rename(columns={"charge_rate_c": "logged_rate_c", "observed_ah": "logged_ah",
                                     "observed_wh": "logged_wh", "charge_minutes": "logged_charge_minutes"})
    replay = []
    for policy, (rate, fallback) in chosen.items():
        decisions = baseline.assign(policy=policy, chosen_rate_c=rate, fallback=fallback)
        decisions["matched"] = decisions.chosen_rate_c.eq(decisions.logged_rate_c)
        decisions["energy_target_hit"] = decisions.logged_wh.ge(decisions.target_wh).where(
            decisions.matched & decisions.logged_wh.notna()).astype("Float64")
        replay.append(decisions)
    return pd.concat(replay, ignore_index=True)

In [6]:
with threadpool_limits(limits=1):
    action_forecasts, calibration = forecast_actions(events, history, cohort, reference)
    decisions = replay_decisions(cohort, action_forecasts, last_energy)
logged_forecasts = action_forecasts.merge(cohort[[*KEYS, "charge_rate_c"]], on=KEYS, validate="many_to_one")
logged_forecasts = logged_forecasts.loc[logged_forecasts.candidate_rate_c.eq(logged_forecasts.charge_rate_c)]
np.testing.assert_allclose(
    logged_forecasts.set_index(KEYS).forecast_wh,
    reference.reindex(logged_forecasts.set_index(KEYS).index).predicted_wh, rtol=0, atol=1e-9)

## Randomized replay and decision gate

Replay uses self-normalized inverse-propensity weights. The documented logging probability is 1/3 for each rate. Each battery gets equal total weight across its eligible origins before matching. Continuous outcomes and the complete-case target-hit estimate use their own measured denominators. The lower and upper target-hit bounds score every missing matched Wh as a miss or as a hit respectively.

The percentiles come from 2,000 paired bootstrap draws of batteries. The same battery draws are used for every policy. The intervals describe replay variation with the fitted policies held fixed. The model advances if charging time is lower with a 95% upper difference bound below zero and the 95% lower bound for its energy hit-rate difference is at least −5 percentage points even under adverse missing-Wh assumptions.

In [7]:
def replay_totals(frame, cell_ids):
    rows_per_cell = frame.groupby("cell_id").start_cycle.transform("size")
    weight = 1 / (LOG_PROBABILITY * rows_per_cell.to_numpy())
    matched = frame.matched.to_numpy()
    hit = frame.energy_target_hit.to_numpy(dtype=float, na_value=np.nan)
    totals = {"match": weight * matched,
              "hit_lower": weight * np.nan_to_num(hit, nan=0),
              "hit_upper": weight * (np.nan_to_num(hit, nan=0) + (matched & ~np.isfinite(hit)))}
    for metric, values in (("hit_rate", hit), ("delivered_wh", frame.logged_wh.to_numpy()),
                           ("charge_minutes", frame.logged_charge_minutes.to_numpy()),
                           ("capacity_ah", frame.logged_ah.to_numpy())):
        available = matched & np.isfinite(values)
        totals[f"{metric}_weight"] = weight * available
        totals[f"{metric}_sum"] = weight * np.where(available, values, 0)
    matrix = pd.DataFrame(totals, index=frame.cell_id).groupby(level=0).sum().reindex(cell_ids)
    if matrix.isna().any().any():
        raise ValueError("Replay totals are missing a battery.")
    return matrix


def replay_estimates(totals, weights):
    totals = pd.DataFrame(weights @ totals.to_numpy(), columns=totals.columns)
    estimates = {}
    for metric in ("hit_rate", "delivered_wh", "charge_minutes", "capacity_ah"):
        estimates[metric] = totals[f"{metric}_sum"] / totals[f"{metric}_weight"]
    estimates["hit_lower"] = totals.hit_lower / totals.match
    estimates["hit_upper"] = totals.hit_upper / totals.match
    result = pd.DataFrame(estimates)
    if not np.isfinite(result.to_numpy()).all():
        raise ValueError("A policy has insufficient replay support in a battery draw.")
    return result


def score_replay(decisions):
    cell_ids = np.sort(decisions.cell_id.unique())
    random = np.random.default_rng(SEED)
    draws = random.multinomial(len(cell_ids), np.full(len(cell_ids), 1 / len(cell_ids)), size=BOOTSTRAP_DRAWS)
    draws = np.vstack((np.ones(len(cell_ids)), draws))
    estimates, counts, intervals, comparisons = {}, [], [], []
    for policy in POLICIES:
        rows = decisions.loc[decisions.policy.eq(policy)]
        matched = rows.loc[rows.matched]
        values = replay_estimates(replay_totals(rows, cell_ids), draws)
        estimates[policy] = values
        counts.append(dict(policy=policy, origins=len(rows), matched=len(matched),
                           matched_batteries=matched.cell_id.nunique(), fallback=int(rows.fallback.sum()),
                           chosen_1c=int(rows.chosen_rate_c.eq(1).sum()),
                           chosen_2c=int(rows.chosen_rate_c.eq(2).sum()),
                           chosen_3c=int(rows.chosen_rate_c.eq(3).sum()),
                           measured_wh=int(matched.logged_wh.notna().sum()),
                           measured_minutes=int(matched.logged_charge_minutes.notna().sum()),
                           measured_ah=int(matched.logged_ah.notna().sum()), **values.iloc[0].to_dict()))
        for metric in values:
            lo, hi = np.quantile(values[metric].iloc[1:], [0.025, 0.975])
            intervals.append(dict(policy=policy, metric=metric, estimate=values.at[0, metric],
                                  ci95_low=lo, ci95_high=hi))

    fixed = estimates["fixed_1c"]
    for policy, values in estimates.items():
        if policy == "fixed_1c":
            continue
        for metric in ("hit_rate", "delivered_wh", "charge_minutes", "capacity_ah"):
            diffs = values[metric] - fixed[metric]
            lo, hi = np.quantile(diffs.iloc[1:], [0.025, 0.975])
            comparisons.append(dict(policy=policy, metric=metric, difference=diffs.iloc[0],
                                    ci95_low=lo, ci95_high=hi, missing_bound_low=np.nan,
                                    missing_bound_high=np.nan))
        worst = values.hit_lower - fixed.hit_upper
        best = values.hit_upper - fixed.hit_lower
        comparisons.append(dict(policy=policy, metric="hit_rate_missing_bounds", difference=np.nan,
                                ci95_low=np.quantile(worst.iloc[1:], 0.025),
                                ci95_high=np.quantile(best.iloc[1:], 0.975),
                                missing_bound_low=worst.iloc[0], missing_bound_high=best.iloc[0]))
    return (pd.DataFrame(counts).set_index("policy"), pd.DataFrame(intervals),
            pd.DataFrame(comparisons))


def assignment_table(cohort):
    result = []
    for label, group in [("all", cohort), *(("previous_" + str(rate), cohort.loc[cohort.previous_rate_c.eq(rate)])
                                          for rate in RATES)]:
        for rate in RATES:
            rows = group.loc[group.charge_rate_c.eq(rate)]
            result.append(dict(group=label, logged_rate_c=rate, logged_count=len(rows),
                               logged_fraction=len(rows) / len(group), missing_next_wh=int(rows.observed_wh.isna().sum()),
                               missing_charge_time=int(rows.charge_minutes.isna().sum())))
    return pd.DataFrame(result)


def acceptance_check(comparisons):
    time = comparisons.set_index(["policy", "metric"]).loc["model", "charge_minutes"]
    energy = comparisons.set_index(["policy", "metric"]).loc["model", "hit_rate_missing_bounds"]
    time_pass = time.ci95_high < 0
    energy_pass = energy.ci95_low >= -ALLOWED_HIT_DROP
    return pd.DataFrame(
        [("time_reduction", time_pass),
         ("energy_noninferiority", energy_pass),
         ("advance_to_simulator", time_pass and energy_pass)], columns=["criterion", "passed"])


with threadpool_limits(limits=1):
    metrics, intervals, comparisons = score_replay(decisions)
    assignment = assignment_table(cohort)
    acceptance = acceptance_check(comparisons)
print(metrics[["matched", "hit_rate", "charge_minutes", "delivered_wh", "capacity_ah"]].round(3).to_string())

          matched  hit_rate  charge_minutes  delivered_wh  capacity_ah
policy                                                                
fixed_1c      294     1.000         186.801        25.375        1.484
fixed_2c      345     0.744          74.500        20.142        1.185
fixed_3c      290     0.142          35.720        14.509        0.857
rule          296     0.761          93.956        19.505        1.150
model         339     1.000         110.309        21.746        1.279


In [8]:
def policy_plot(path, metrics, intervals):
    figure, axes = plt.subplots(1, 2, figsize=(10.5, 3.9), layout="constrained")
    for ax, metric, label, scale in zip(
        axes, ("hit_rate", "charge_minutes"),
        ("Energy target met (%)", "Charging time for five cycles (min)"), (100, 1)
    ):
        band = intervals.loc[intervals.metric.eq(metric)].set_index("policy").reindex(POLICIES)
        for position, policy in enumerate(POLICIES):
            row = band.loc[policy]
            color = "#b93834" if policy == "model" else "#235789"
            ax.hlines(position, row.ci95_low * scale, row.ci95_high * scale, color=color, linewidth=2)
            ax.scatter(row.estimate * scale, position, color=color, zorder=3)
        ax.set_yticks(range(len(POLICIES)), POLICIES.values())
        ax.invert_yaxis()
        ax.set_xlabel(label)
        ax.grid(axis="x", alpha=0.20)
        if metric == "hit_rate":
            ax.axvline((metrics.at["fixed_1c", "hit_rate"] - ALLOWED_HIT_DROP) * 100,
                       color="#62676d", linestyle="--", linewidth=1)
    figure.suptitle("MIT–BIT held-out charging policies · 95% battery bootstrap intervals")
    figure.savefig(path, dpi=150)
    plt.close(figure)


def summary_text(metrics, comparisons, acceptance, cohort, assignment, charge_audit):
    model = metrics.loc["model"]
    fixed = metrics.loc["fixed_1c"]
    diffs = comparisons.set_index(["policy", "metric"])
    time = diffs.loc["model", "charge_minutes"]
    hit = diffs.loc["model", "hit_rate"]
    bound = diffs.loc["model", "hit_rate_missing_bounds"]
    checks = acceptance.set_index("criterion").passed
    gate = ("qualified for the next simulator experiment" if checks.advance_to_simulator
            else "fell short of the criteria for the next simulator experiment")
    timing = "passed" if checks.time_reduction else "failed"
    energy = "passed" if checks.energy_noninferiority else "failed"
    lines = [
        "# MIT–BIT charging controller benchmark", "",
        "## Results", "",
        f"We compared five charging policies across **{len(cohort)} decisions on {cohort.cell_id.nunique()} "
        f"held-out batteries**. For each five-cycle block, the goal was to deliver at least "
        f"{PREVIOUS_ENERGY_FRACTION:.0%} of the energy measured in the preceding block.",
        "",
        "| Policy | Energy target met | Charging time (min per 5 cycles) | Delivered energy (Wh per 5 cycles) | Discharge capacity (Ah)¹ | Matched decisions |",
        "| --- | ---: | ---: | ---: | ---: | ---: |",
    ]
    for policy, row in metrics.iterrows():
        lines.append(f"| {POLICIES[policy]} | {row.hit_rate:.1%} | {row.charge_minutes:.1f} | "
                     f"{row.delivered_wh:.2f} | {row.capacity_ah:.3f} | {int(row.matched)}/{int(row.origins)} |")
    lines.extend([
        "",
        "¹ Weighted average of the median discharge capacity measured within each five-cycle block.", "",
        f"**The model-guided policy {gate}.** Compared with fixed 1C, its mean charging time differed by "
        f"**{time.difference:+.1f} minutes** per block (95% CI {time.ci95_low:+.1f} to {time.ci95_high:+.1f}). "
        f"The energy target hit rate differed by **{hit.difference * 100:+.1f} percentage points** "
        f"(95% CI {hit.ci95_low * 100:+.1f} to {hit.ci95_high * 100:+.1f}). "
        f"The policy {timing} the charging-time criterion and {energy} the energy criterion, "
        f"which allows a drop of up to {ALLOWED_HIT_DROP * 100:g} percentage points in the hit rate.",
        "",
        f"Relative to fixed 1C, charging time changed by {time.difference / fixed.charge_minutes:+.1%} "
        f"and delivered energy changed by {model.delivered_wh - fixed.delivered_wh:+.2f} Wh per five-cycle block. "
        "The reported hit rates use matched decisions with measured delivered energy.",
        "",
        f"The model-guided policy chose 1C in {int(model.chosen_1c)} decisions, 2C in {int(model.chosen_2c)}, "
        f"and 3C in {int(model.chosen_3c)}. It used its fallback in {int(model.fallback)} decisions.",
        "",
        "## Evaluation details", "",
        f"Eligible blocks start at cycles {FIRST_START}–{LAST_START} and require a measured previous block. "
        f"There were {int(cohort.observed_wh.isna().sum())} missing next-block Wh outcomes overall. "
        f"Fixed 1C had {int(fixed.matched - fixed.measured_wh)} matched blocks with missing Wh. "
        f"The model had {int(model.matched - model.measured_wh)}. "
        f"Bounding these missing outcomes gives a 95% difference interval of "
        f"{bound.ci95_low * 100:+.1f} to {bound.ci95_high * 100:+.1f} percentage points.",
        "",
        f"Replay uses exact action matches, the documented 1/3 assignment probability and normalized weights. "
        f"Each battery receives equal total weight before matching. Recorded assignment counts "
        f"for 1C, 2C and 3C were {', '.join(str(n) for n in assignment.loc[assignment.group.eq('all'), 'logged_count'])}. "
        f"Timing is the sum of five measured charging intervals, available for "
        f"{cohort.charge_minutes.notna().sum()}/{len(cohort)} eligible blocks. "
        f"For {charge_audit.status.eq('ok_test_time').sum()} cycles with a wall-clock discontinuity, "
        f"the recorder's elapsed Test_Time supplies the duration. Wh is the sum of five meter-checked discharges. "
        f"Bootstrap intervals use {BOOTSTRAP_DRAWS:,} paired battery draws with the fitted policies held fixed.",
        "",
        "The benchmark measures a single decision on recorded randomized histories in the same MIT–BIT cohort "
        "used in Notebook 15. Evaluating battery lifetime under repeated policy decisions requires a separate experiment.",
        "",
        "[MIT–BIT v3 data and experimental design](https://data.mendeley.com/datasets/kw34hhw7xg/3)",
        "",
    ])
    return "\n".join(lines)


output.mkdir(parents=True, exist_ok=True)
tables = {
    "policy_metrics": metrics.reset_index(), "confidence_intervals": intervals, "comparisons_vs_fixed_1c": comparisons,
    "acceptance": acceptance, "decisions": decisions, "action_forecasts": action_forecasts, "calibration": calibration,
    "assignment_audit": assignment, "charge_time_audit": charge_audit,
}
for name, table in tables.items():
    table.to_csv(output / f"{name}.csv", index=False)
config = dict(dataset_doi="10.17632/kw34hhw7xg.3", batteries=cohort.cell_id.nunique(), eligible_origins=len(cohort),
              start_cycle_window=[FIRST_START, LAST_START], block_size=BLOCK, logging_probability=LOG_PROBABILITY,
              energy_target_fraction=PREVIOUS_ENERGY_FRACTION, overprediction_quantile=OVERPREDICTION_QUANTILE,
              permitted_hit_rate_drop=ALLOWED_HIT_DROP, bootstrap_draws=BOOTSTRAP_DRAWS, seed=SEED,
              model="0.75 standard Ridge(alpha=1) + 0.25 expanded Ridge(alpha=1)")
(output / "run_config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
(output / "summary.md").write_text(summary_text(metrics, comparisons, acceptance, cohort, assignment, charge_audit), encoding="utf-8")
policy_plot(output / "policy_comparison.png", metrics, intervals)
archive_path = output.with_suffix(".zip")
names = [*(f"{name}.csv" for name in tables), "run_config.json", "summary.md", "policy_comparison.png"]
with ZipFile(archive_path, "w", ZIP_DEFLATED) as bundle:
    for name in names:
        bundle.write(output / name, arcname=f"{output.name}/{name}")
print(acceptance.to_string(index=False))
print(f"Report: {output / 'summary.md'}")
print(f"Download: {archive_path}")

            criterion  passed
       time_reduction    True
energy_noninferiority    True
 advance_to_simulator    True
Report: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_mit_bit_controller_benchmark_results/summary.md
Download: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_mit_bit_controller_benchmark_results.zip
